In [1]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip


In [2]:
STRING_COLS = {
    "usuarios":    ["nome", "email", "pais"],
    "planos":      ["nome"],
    "artistas":    ["nome", "pais"],
    "albuns":      ["titulo"],
    "musicas":     ["titulo"],
    "generos":     ["nome"],
    "playlists":   ["nome"],
    "dispositivos":["tipo", "so"],
}


In [3]:
load_dotenv()

is_docker = os.path.exists("/.dockerenv")
MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

builder = (
    SparkSession.builder
    .appName("silver-dimensoes")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.defaultFS", "file:///")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST, access_key=MINIO_USER, secret_key=MINIO_PASS, secure=False
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 17:05:51 WARN Utils: Your hostname, DESKTOP-S3JB25M, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/21 17:05:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/c/Users/felip/www/university/eng-dados/projeto-ed-final/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/felipe/.ivy2.5.2/cache
The jars for the packages stored in: /home/felipe/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-24c3ca4c-bef5-4d65-af22-78203251bf5f;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf

In [4]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets:", buckets)
if "silver" not in buckets:
    cliente_minio.make_bucket("silver")
    print("Bucket 'silver' criado.")


Buckets: {'landing', 'gold', 'silver', 'bronze'}


In [5]:
def download_delta_do_minio(bucket: str, prefix: str,
                            cliente: Minio, local_dir: Path) -> None:
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True)
    objetos = list(cliente.list_objects(bucket, prefix=f"{prefix}/", recursive=True))
    print(f"  {len(objetos)} arquivo(s) em {bucket}/{prefix}/")
    for obj in objetos:
        relative = obj.object_name[len(prefix) + 1:]
        destino = local_dir / relative
        destino.parent.mkdir(parents=True, exist_ok=True)
        cliente.fget_object(bucket, obj.object_name, str(destino))


def upload_delta_para_minio(local_dir: Path, bucket: str, prefix: str,
                            cliente: Minio) -> None:
    for arq in sorted(local_dir.rglob("*")):
        if arq.is_file():
            object_name = f"{prefix}/{arq.relative_to(local_dir)}"
            cliente.fput_object(bucket, object_name, str(arq))
            print(f"  → {object_name}")


In [6]:
for tabela, cols_str in STRING_COLS.items():
    print(f"\n{'=' * 55}")
    print(f"Silver: {tabela}")
    print(f"{'=' * 55}")

    local_bronze = Path(f"/tmp/bronze_local/{tabela}")
    download_delta_do_minio("bronze", f"dimensoes/{tabela}", cliente_minio, local_bronze)
    df = spark.read.format("delta").load(str(local_bronze))
    total_bronze = df.count()
    print(f"Bronze lido: {total_bronze:,} registros")

    df = df.dropDuplicates(["id"])
    removidos = total_bronze - df.count()
    if removidos:
        print(f"Duplicatas removidas: {removidos:,}")

    for col in cols_str:
        df = df.withColumn(col, F.trim(F.col(col)))

    local_silver = Path(f"/tmp/silver/{tabela}")
    if local_silver.exists():
        shutil.rmtree(local_silver)

    df.write.format("delta").mode("overwrite").save(str(local_silver))
    print(f"Silver Delta escrito em {local_silver}")

    print(f"Enviando para MinIO silver/dimensoes/{tabela}/")
    upload_delta_para_minio(local_silver, "silver", f"dimensoes/{tabela}", cliente_minio)
    print(f"{tabela} Silver concluído — {df.count():,} registros.")

print("\nTransformação Silver de todas as dimensões concluída.")



Silver: usuarios
  6 arquivo(s) em bronze/dimensoes/usuarios/


26/06/21 17:06:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze lido: 3,000 registros


Silver Delta escrito em /tmp/silver/usuarios
Enviando para MinIO silver/dimensoes/usuarios/
  → dimensoes/usuarios/.part-00000-9a3facb1-68a7-480d-8432-1097a3f8522f-c000.snappy.parquet.crc
  → dimensoes/usuarios/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/usuarios/_delta_log/.00000000000000000000.json.crc
  → dimensoes/usuarios/_delta_log/00000000000000000000.crc
  → dimensoes/usuarios/_delta_log/00000000000000000000.json
  → dimensoes/usuarios/part-00000-9a3facb1-68a7-480d-8432-1097a3f8522f-c000.snappy.parquet
usuarios Silver concluído — 3,000 registros.

Silver: planos
  6 arquivo(s) em bronze/dimensoes/planos/


Bronze lido: 3 registros


...

Silver Delta escrito em /tmp/silver/artistas
Enviando para MinIO silver/dimensoes/artistas/
  → dimensoes/artistas/.part-00000-509d9e40-f6fb-49b0-9d9a-cc631515ce51-c000.snappy.parquet.crc
  → dimensoes/artistas/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/artistas/_delta_log/.00000000000000000000.json.crc
  → dimensoes/artistas/_delta_log/00000000000000000000.crc
  → dimensoes/artistas/_delta_log/00000000000000000000.json
  → dimensoes/artistas/part-00000-509d9e40-f6fb-49b0-9d9a-cc631515ce51-c000.snappy.parquet
artistas Silver concluído — 500 registros.

Silver: albuns
  6 arquivo(s) em bronze/dimensoes/albuns/


Bronze lido: 2,000 registros
Silver Delta escrito em /tmp/silver/albuns
Enviando para MinIO silver/dimensoes/albuns/
  → dimensoes/albuns/.part-00000-80d631cd-9a26-443c-b1f4-5c0c2bba152b-c000.snappy.parquet.crc
  → dimensoes/albuns/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/albuns/_delta_log/.00000000000000000000.json.crc
  → dimensoes/albuns/_delta_log/00000000000000000000.crc
  → dimensoes/albuns/_delta_log/00000000000000000000.json
  → dimensoes/albuns/part-00000-80d631cd-9a26-443c-b1f4-5c0c2bba152b-c000.snappy.parquet
albuns Silver concluído — 2,000 registros.

Silver: musicas
  6 arquivo(s) em bronze/dimensoes/musicas/


Bronze lido: 10,000 registros
Silver Delta escrito em /tmp/silver/musicas
Enviando para MinIO silver/dimensoes/musicas/
  → dimensoes/musicas/.part-00000-c987ccfd-ec74-4554-bbc1-a8b94a55c64a-c000.snappy.parquet.crc
  → dimensoes/musicas/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/musicas/_delta_log/.00000000000000000000.json.crc
  → dimensoes/musicas/_delta_log/00000000000000000000.crc
  → dimensoes/musicas/_delta_log/00000000000000000000.json
  → dimensoes/musicas/part-00000-c987ccfd-ec74-4554-bbc1-a8b94a55c64a-c000.snappy.parquet
musicas Silver concluído — 10,000 registros.

Silver: generos
  6 arquivo(s) em bronze/dimensoes/generos/


Bronze lido: 30 registros
Silver Delta escrito em /tmp/silver/generos
Enviando para MinIO silver/dimensoes/generos/
  → dimensoes/generos/.part-00000-3de8c730-e022-4374-bf66-acbd2c84f8f2-c000.snappy.parquet.crc
  → dimensoes/generos/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/generos/_delta_log/.00000000000000000000.json.crc
  → dimensoes/generos/_delta_log/00000000000000000000.crc
  → dimensoes/generos/_delta_log/00000000000000000000.json
  → dimensoes/generos/part-00000-3de8c730-e022-4374-bf66-acbd2c84f8f2-c000.snappy.parquet
generos Silver concluído — 30 registros.

Silver: playlists
  6 arquivo(s) em bronze/dimensoes/playlists/
Bronze lido: 2,000 registros
Silver Delta escrito em /tmp/silver/playlists
Enviando para MinIO silver/dimensoes/playlists/
  → dimensoes/playlists/.part-00000-86ff2d3e-2ce2-4e48-9614-01fbb89f1593-c000.snappy.parquet.crc
  → dimensoes/playlists/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/playlists/_delta_log/.00000000000000000000.json.cr

Bronze lido: 5,000 registros
Silver Delta escrito em /tmp/silver/dispositivos
Enviando para MinIO silver/dimensoes/dispositivos/
  → dimensoes/dispositivos/.part-00000-315cfbf9-49f3-4a94-a59b-8072f7e57efc-c000.snappy.parquet.crc
  → dimensoes/dispositivos/_delta_log/.00000000000000000000.crc.crc
  → dimensoes/dispositivos/_delta_log/.00000000000000000000.json.crc
  → dimensoes/dispositivos/_delta_log/00000000000000000000.crc
  → dimensoes/dispositivos/_delta_log/00000000000000000000.json
  → dimensoes/dispositivos/part-00000-315cfbf9-49f3-4a94-a59b-8072f7e57efc-c000.snappy.parquet
dispositivos Silver concluído — 5,000 registros.

Transformação Silver de todas as dimensões concluída.


In [7]:
spark.stop()
